# Comparison among refinement methods — MindEye2 vs ControlNet

Merging tables from `final_evaluations.ipynb` for different methods
(baseline, SDXL enhanced, ControlNet, ...) in a single comparison.

**Prerequisite:** run `final_evaluations.ipynb` once for each method, passing a different `--all_recons_path` every time. Every run saves a table in `tables/{model_name}_<suffix>.csv`.

```bash
python final_evaluations.py --all_recons_path=evals/.../..._all_recons.pt
python final_evaluations.py --all_recons_path=evals/.../..._all_enhancedrecons.pt --blend_blurry
python final_evaluations.py --all_recons_path=evals/.../..._all_controlnet_canny.pt
```

**Note:** "Captions evaluation" section from `final_evaluations.ipynb`
always evaluates the same captions (`all_predcaptions.pt`, generated by GIT
during `recon_inference`), independent of which `--all_recons_path`
was passed — does not distinguish among refinement methods.
This notebook focuses on image quality metrics (PixCorr,
SSIM, AlexNet, InceptionV3, CLIP, EffNet-B, SwAV, retrieval, brain corr.).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def is_interactive():
    import __main__
    return not hasattr(__main__, '__file__')


In [ ]:

model_name = 'final_subj01_pretrained_40sess_24bs'
tables_dir = 'tables'

methods = {
    'MindEye2 (baseline)': 'all_recons',
    'SDXL Enhanced':       'all_enhancedrecons',
    'ControlNet (Canny)':  'all_controlnet_canny',
  
}

# Reference method to compute relative improvement
baseline_label = 'MindEye2 (baseline)'

# EffNet-B and SwAV -> the lower, the better
LOWER_IS_BETTER = {'EffNet-B', 'SwAV'}


In [ ]:
# Load and merge tables per method
frames, missing = [], []
for label, suffix in methods.items():
    path = f'{tables_dir}/{model_name}_{suffix}.csv'
    if not os.path.exists(path):
        missing.append(path)
        continue
    df = pd.read_csv(path, sep=',')
    df['Method'] = label
    frames.append(df)

if missing:
    print('WARNING: tables not foung . Run final_evaluations.ipynb before:')
    for m in missing:
        print(f'  - {m}')

assert frames, 'No table found. Run final_evaluations.ipynb for at least one method.'

long_df = pd.concat(frames, ignore_index=True)
wide_df = long_df.pivot(index='Metric', columns='Method', values='Value')
wide_df = wide_df[[m for m in methods if m in wide_df.columns]] 
print(wide_df)


In [ ]:
# For "the lower the better" metrics invert delta sign
comparison = wide_df.copy()

if baseline_label in comparison.columns:
    for label in wide_df.columns:
        if label == baseline_label:
            continue
        delta_pct = (comparison[label] - comparison[baseline_label]) / comparison[baseline_label].abs() * 100
        sign = comparison.index.map(lambda m: -1 if m in LOWER_IS_BETTER else 1)
        comparison[f'{label} - Δ% vs baseline'] = (delta_pct * sign).round(2)

print(comparison)


In [ ]:
# Saving
os.makedirs(tables_dir, exist_ok=True)
out_path = f'{tables_dir}/{model_name}_method_comparison.csv'
comparison.to_csv(out_path, sep=',')
print(f'Saved: {out_path}')


In [ ]:
# Bar chart
metrics = list(wide_df.index)
n = len(metrics)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = np.array(axes).flatten()

colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))

for ax, metric in zip(axes, metrics):
    values = wide_df.loc[metric]
    ax.bar(values.index, values.values, color=colors[:len(values)])
    direction = '↓ better' if metric in LOWER_IS_BETTER else '↑ better'
    ax.set_title(f'{metric}\n({direction})', fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

for ax in axes[n:]:
    ax.axis('off')

fig.suptitle(f'Methods comparison — {model_name}', fontsize=13)
fig.tight_layout()
fig.savefig(f'{tables_dir}/{model_name}_method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Merge caption metrics tables
caption_frames = []
for label, suffix in methods.items():
    path = f'{tables_dir}/{model_name}_{suffix}_caption_metrics.csv'
    if os.path.exists(path):
        df = pd.read_csv(path, sep=',')
        df['Method'] = label
        caption_frames.append(df)

if caption_frames:
    cap_long = pd.concat(caption_frames, ignore_index=True)
    cap_wide = cap_long.pivot(index='Metric', columns='Method', values='Value')
    print(cap_wide)
else:
    print('No caption metrics table found ...') 
